# Quick start

This notebook is a basic quickstart guide to the code used for this project.

In [ ]:
from rdkit import Chem
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import topological_pretraining as tp

## Molecular Standardizer

In our work, all molecules were standardized using rdkit. The standardizer is usable as follows:

In [ ]:
# Example of using the Standardizer to standardize a molecule
# here standardization alters the tautomer
example_mol = Chem.MolFromSmiles("C=C(O)O")
standardizer = tp.data.mol.Standardizer()
standardized_mol = standardizer(example_mol)
Chem.Draw.MolsToGridImage([example_mol, standardized_mol], legends=["Original", "Standardized"])

In [ ]:
# Example of molecule that fails standardization
# Here we santize the molecule internally to produce a more verbose error message
fail_mol = Chem.MolFromSmiles("NC(=O)NC1N=C(O[AlH3](O)O)NC1=O", sanitize=False)
standardized_fail_mol = standardizer(fail_mol)
standardized_fail_mol

In [ ]:
# standardizer options from docs
help(tp.data.mol.Standardizer)

## Dataset loading

Datasets were stored as custom pandas dataframes with additional attributes for rdkit molecules, standardizers etc. For more detailed explainations, see the [datasets notebook](./01_datasets.ipynb).

Dataset loading can be done using `topological_pretraining.data.load_dataset`. If downloade, SMILES are converted into rdkit molecules and standardized. `load_dataset` takes the following args:
- `name: str` a string naming the dataset; the available datasets are listed in `topological_pretraining.data.available_datasets`.
- `root: str | None`: an optional dir to save or load from; if no dir is provided the dataset is downloaded from a pre-defined url and not saved to disk.
- `compression: bool`: whether to save the dataset using gzip. Default is `True`.
- `verbose: bool`: whether to print loading and processing statements. Default is `False`.
- `standardizer: topological_pretraining.data.mol.Standardizer` a standardizer object for preprocessing molecules.

In [ ]:
# Loading Rat_PPB dataset from online with a standardizer
standardizer = tp.data.mol.Standardizer(canonical_tautomer=True)
df = tp.data.utils.load_dataset("Rat_PPB", standardizer=standardizer, verbose=True)

In [ ]:
# the dataset inherits from pandas DataFrame so you can use all the usual pandas methods
df.head()

## Using a pre-trained model

We generally recommend pre-training your own models on data and tasks (which may include pre-training on ECFPs) that are tailored to your downstream task. We don't necessarily recommend using our models pre-trained on QMugs, as:
- there may be data leakage between our pre-training data and your task data;
- these models were deliberately pre-trained on subsets of QMugs (~465k molecules) to answer specific experimental questions, and are not foundation models.

However, you can use our models pre-trained on QMugs if you would like to.

In [ ]:
# path to a GIN pre-trained on QMugs with 0.5 tanimoto similarity threshold filtering
# for pre-training, any molecule with a tanimoto similarity above 0.5 to any molecule 
# in the downstream dataset was removed from the pre-training dataset
# with radius 1 and vocab size 2048 for substructure tokenization
model_path = "../pt_models/vocab_size/pt_gin_radius_1_vocab_2048.pt"

In [ ]:
model = tp.featurization.PreTrainedFeaturizer(model_path)
model

In [ ]:
# featurize an rdkit molecules from Rat_PPB with the pre-trained model
# ignores heads
X = model(df.rdkit_mols)
X.shape

In [ ]:
# get the target variable
y = df.y
y

In [ ]:
# basic random train test split
train, test = train_test_split(range(len(X)), test_size=0.2, shuffle=True)

In [ ]:
lgbm = LGBMRegressor(
    random_state=42, n_jobs=-1
)
lgbm.fit(X[train], y[train])

In [ ]:
preds = lgbm.predict(X[test])
mean_squared_error(y[test], preds)